# Fire Risk Prediction — Exploration & Démonstration

Bassin méditerranéen (35°N–47°N, 5°W–28°E) · 2018–2023 · Grille 0.25°

Ce notebook guide l'utilisateur à travers l'ensemble du pipeline :
1. Fusion des données (pipeline.py)
2. Dataset & Dataloaders
3. Architecture du modèle
4. Entraînement rapide
5. Évaluation
6. Visualisations EDA

In [ ]:
import sys, os
from pathlib import Path

# Ajouter src/ au path
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Project : {project_root}')

: 

## 1. Pipeline — Fusion des données (mode simulé)

In [ ]:
from pipeline import run_pipeline

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'

# Simulation rapide sur 60 jours pour la démo
import pipeline as pl
import pandas as pd
import numpy as np

# Simulation manuelle sur une courte période
pl.LATS = np.arange(35, 38, 0.25)
pl.LONS = np.arange(-5, -2, 0.25)
pl.N_LAT = len(pl.LATS)
pl.N_LON = len(pl.LONS)

print(f'Grille démo : {pl.N_LAT} × {pl.N_LON} = {pl.N_LAT * pl.N_LON} cellules')
print('Pipeline complet → voir src/pipeline.py pour la vraie exécution')

## 2. Dataset & Dataloaders

In [ ]:
from dataset import SimulatedFireRiskDataset, build_dataloaders, FEATURE_NAMES, SEQ_LEN

# Dataset simulé
train_ds = SimulatedFireRiskDataset(n_days=300, n_lat=10, n_lon=10, is_train=True, seed=0)
val_ds   = SimulatedFireRiskDataset(n_days=100, n_lat=10, n_lon=10,
                                     scaler=train_ds.scaler, seed=1)

x, y = train_ds[0]
print(f'Features    : {FEATURE_NAMES}')
print(f'x.shape     : {x.shape}  (seq_len={SEQ_LEN}, n_features={len(FEATURE_NAMES)})')
print(f'y           : {y.item():.4f}')
print(f'Train set   : {len(train_ds):,} séquences')
print(f'Val set     : {len(val_ds):,} séquences')
print(f'Risque élevé (>0.5) : {(train_ds.y > 0.5).mean() * 100:.1f}%')

## 3. Architectures des modèles

In [ ]:
from model import build_model, FireRiskLSTM, FireRiskTCN

lstm = build_model('lstm', n_features=8)
tcn  = build_model('tcn',  n_features=8)

# Test forward pass
x_batch = torch.randn(16, SEQ_LEN, 8)
out_lstm = lstm(x_batch)
out_tcn  = tcn(x_batch)

print(f'LSTM output : {out_lstm.shape}  min={out_lstm.min():.3f}  max={out_lstm.max():.3f}')
print(f'TCN  output : {out_tcn.shape}   min={out_tcn.min():.3f}  max={out_tcn.max():.3f}')
print(lstm)

## 4. Entraînement rapide (5 époques demo)

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from train import train, set_seed, weighted_mse_loss

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Samplers
weights = torch.from_numpy(train_ds.get_sample_weights())
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=256, sampler=sampler)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)

# Entraînement rapide
model = build_model('lstm', n_features=8)
history = train(
    model, train_loader, val_loader,
    output_dir=processed_dir,
    n_epochs=5,
    patience=5,
    device=device,
    model_name='lstm_demo',
)
print('Entraînement terminé !')

In [ ]:
# Courbes de loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train', linewidth=2)
ax.plot(history['val_loss'],   label='Val',   linewidth=2)
ax.set(xlabel='Époque', ylabel='Weighted MSE', title='Courbes de loss LSTM')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Évaluation

In [ ]:
from evaluate import predict, compute_mae, compute_rmse, compute_spearman, compute_auc

test_ds = SimulatedFireRiskDataset(n_days=80, n_lat=10, n_lon=10,
                                    scaler=train_ds.scaler, seed=99)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

y_true, y_pred = predict(model, test_loader, device)

print(f'MAE      : {compute_mae(y_true, y_pred):.4f}')
print(f'RMSE     : {compute_rmse(y_true, y_pred):.4f}')
print(f'Spearman : {compute_spearman(y_true, y_pred):.4f}')
print(f'AUC      : {compute_auc(y_true, y_pred):.4f}')

In [ ]:
# Scatter plot
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_true[:500], y_pred[:500], alpha=0.4, s=8, c='#e63946')
ax.plot([0,1],[0,1],'k--', linewidth=1.5)
ax.set(xlabel='Risque réel', ylabel='Risque prédit', title='Réel vs Prédit')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 6. Visualisations EDA

In [ ]:
from visualize import simulate_eda_data, run_eda

figures_dir = project_root / 'data' / 'figures'
df_sim = simulate_eda_data(n_cells=2000, seed=42)
run_eda(df_sim, figures_dir)
print('Figures EDA générées dans data/figures/')

In [ ]:
# Afficher la carte de risque
from visualize import plot_risk_map_mediterranean, N_LAT, N_LON, LATS, LONS
import numpy as np

rng = np.random.default_rng(42)
summer = np.sin(np.pi * np.linspace(0, 1, N_LAT))[:, None]
risk_grid = np.clip(0.1 + 0.5 * summer + rng.normal(0, 0.07, (N_LAT, N_LON)), 0, 1)

# Affichage inline
import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('fire', ['#2b83ba','#ffffbf','#fdae61','#d7191c'])
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.pcolormesh(LONS, LATS, risk_grid, cmap=cmap, vmin=0, vmax=1, shading='auto')
plt.colorbar(im, ax=ax, label='Risque ∈ [0,1]', shrink=0.8)
ax.set(xlabel='Longitude (°E)', ylabel='Latitude (°N)',
       title='Risque incendie — Bassin méditerranéen (simulé)')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout(); plt.show()